# Secure Data Sharing: Granularity Demo

This notebook shows that Snowflake Secure Data Sharing is **not limited to whole databases or schemas**. A provider decides exactly what a consumer can see, at these levels:

| # | Level | Mechanism | Object in this demo |
|---|---|---|---|
| 1 | Schema | Only grant `USAGE` on the schemas you want to expose | `HR` schema is never granted |
| 2 | Table | Grant `SELECT` per table | `SALES.ORDERS` shared, `SALES.INTERNAL_COSTS` not |
| 3 | Column | Secure view that projects a subset of columns | `SHARED.V_CUSTOMERS_NO_PII` |
| 4 | Row (static) | Secure view with a `WHERE` filter | `SHARED.V_ORDERS_APAC` |
| 5 | Row (per consumer) | Secure view checked against an entitlement table on `CURRENT_ACCOUNT()` | `SHARED.V_ORDERS_ENTITLED` |

**Provider:** `<PROVIDER_ORG>.<PROVIDER_ACCOUNT>`, the account running this notebook  
**Consumer:** `<CONSUMER_ORG>.<CONSUMER_ACCOUNT>`, which must be in the **same region** as the provider for a direct share. (For another region, use a listing with auto-fulfillment.)

Set the consumer values in the configuration cell in Step 0 before running.

No data is copied. The consumer queries the provider's storage live and read-only, using its own compute.

## Step 0: Context and configuration
Creating shares needs the `CREATE SHARE` account privilege. `ACCOUNTADMIN` has it by default.

Set `CONSUMER_ACCOUNT` and `CONSUMER_LOCATOR` in the next cell. The consumer's locator is what `CURRENT_ACCOUNT()` returns when run in the consumer account.

In [ ]:
# Set these before running. The SQL cells below read them through Jinja ({{ ... }}).
# CONSUMER_ACCOUNT: consumer account identifier in <ORG_NAME>.<ACCOUNT_NAME> form
# CONSUMER_LOCATOR: consumer account locator (run SELECT CURRENT_ACCOUNT(); in the consumer account)
import re

CONSUMER_ACCOUNT = "<CONSUMER_ORG>.<CONSUMER_ACCOUNT>"
CONSUMER_LOCATOR = "<CONSUMER_LOCATOR>"

# Values are substituted into SQL text, so allow only valid identifier characters
assert re.fullmatch(r"[A-Za-z0-9_]+\.[A-Za-z0-9_]+", CONSUMER_ACCOUNT), "Set CONSUMER_ACCOUNT to <ORG_NAME>.<ACCOUNT_NAME>"
assert re.fullmatch(r"[A-Za-z0-9_]+", CONSUMER_LOCATOR), "Set CONSUMER_LOCATOR to the consumer's account locator"
CONSUMER_ACCOUNT, CONSUMER_LOCATOR = CONSUMER_ACCOUNT.upper(), CONSUMER_LOCATOR.upper()

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;
SELECT CURRENT_ROLE() AS role, CURRENT_ORGANIZATION_NAME() AS org, CURRENT_ACCOUNT_NAME() AS account, CURRENT_REGION() AS region;

## Step 1: Create the dummy database
Three schemas:
- `SALES`: business data, **partially** shared
- `HR`: sensitive, **never** shared
- `SHARED`: secure views and the entitlement table that make up the curated sharing layer

In [ ]:
%%sql -r create_db
-- Idempotent: IF NOT EXISTS never drops existing objects (and so never breaks share grants)
CREATE DATABASE IF NOT EXISTS SHARING_DEMO_DB COMMENT = 'Secure data sharing granularity demo';
CREATE SCHEMA IF NOT EXISTS SHARING_DEMO_DB.SALES;
CREATE SCHEMA IF NOT EXISTS SHARING_DEMO_DB.HR;
CREATE SCHEMA IF NOT EXISTS SHARING_DEMO_DB.SHARED;

In [ ]:
%%sql -r load_sales
-- Idempotent: CREATE IF NOT EXISTS keeps the table (and its share grants);
-- INSERT OVERWRITE replaces the rows, so re-runs never duplicate data
CREATE TABLE IF NOT EXISTS SHARING_DEMO_DB.SALES.CUSTOMERS (
  CUSTOMER_ID INT, CUSTOMER_NAME STRING, EMAIL STRING, PHONE STRING, COUNTRY STRING, SEGMENT STRING
);
INSERT OVERWRITE INTO SHARING_DEMO_DB.SALES.CUSTOMERS VALUES
 (1,'Acme Pte Ltd','ops@acme.sg','+65 6111 1111','Singapore','Enterprise'),
 (2,'Kangaroo Corp','info@kangaroo.au','+61 2 9222 2222','Australia','Mid-Market'),
 (3,'Sakura KK','hello@sakura.jp','+81 3 3333 3333','Japan','Enterprise'),
 (4,'Bavaria GmbH','kontakt@bavaria.de','+49 89 444444','Germany','SMB'),
 (5,'Thames Ltd','hi@thames.co.uk','+44 20 5555 5555','United Kingdom','Mid-Market'),
 (6,'Golden Gate Inc','sales@goldengate.com','+1 415 666 6666','United States','Enterprise');

CREATE TABLE IF NOT EXISTS SHARING_DEMO_DB.SALES.ORDERS (
  ORDER_ID INT, CUSTOMER_ID INT, REGION STRING, ORDER_DATE DATE, AMOUNT_USD NUMBER(12,2)
);
INSERT OVERWRITE INTO SHARING_DEMO_DB.SALES.ORDERS VALUES
 (101,1,'APAC','2026-01-05',12000),(102,2,'APAC','2026-01-17',5400),(103,3,'APAC','2026-02-02',22000),
 (104,4,'EMEA','2026-02-10',3100),(105,5,'EMEA','2026-03-01',7800),(106,6,'AMER','2026-03-12',15500),
 (107,1,'APAC','2026-04-08',9900),(108,6,'AMER','2026-04-20',4300),(109,4,'EMEA','2026-05-15',6200);

CREATE TABLE IF NOT EXISTS SHARING_DEMO_DB.SALES.INTERNAL_COSTS (
  ORDER_ID INT, COST_USD NUMBER(12,2), MARGIN_PCT NUMBER(5,2)
);
INSERT OVERWRITE INTO SHARING_DEMO_DB.SALES.INTERNAL_COSTS VALUES
 (101,7000,41.67),(102,3900,27.78),(103,12500,43.18),(104,2600,16.13),(105,5100,34.62),
 (106,9800,36.77),(107,6100,38.38),(108,3500,18.60),(109,4400,29.03);

In [ ]:
%%sql -r load_hr
CREATE TABLE IF NOT EXISTS SHARING_DEMO_DB.HR.EMPLOYEES (
  EMPLOYEE_ID INT, FULL_NAME STRING, SALARY_USD NUMBER(12,2), NATIONAL_ID STRING
);
INSERT OVERWRITE INTO SHARING_DEMO_DB.HR.EMPLOYEES VALUES
 (1,'Alice Tan',120000,'S1234567A'),(2,'Bob Lee',95000,'S7654321B'),(3,'Chen Wei',150000,'S1111111C');

## Step 2: Build the curated sharing layer (secure views)
**Secure views** hide the view definition from the consumer and block optimizer-based inference of filtered-out data, so they are a safe way to share **subsets** of a table.

- **Column level:** `V_CUSTOMERS_NO_PII` leaves out `EMAIL` and `PHONE`.
- **Static row level:** `V_ORDERS_APAC` returns only `REGION = 'APAC'`.
- **Per-consumer row level:** `V_ORDERS_ENTITLED` checks `CONSUMER_ENTITLEMENTS`. When a shared view runs in the consumer account, `CURRENT_ACCOUNT()` returns the **consumer's account locator**, so one share can show different rows to each consumer. The table is keyed on the locator because that is what `CURRENT_ACCOUNT()` returns, and it is also what the provider-side simulation in Step 6 overrides. The provider's own locator is filled in automatically with `CURRENT_ACCOUNT()`.

The entitlement table itself is **not** shared, so consumers can't see who else is entitled to what.

In [ ]:
-- Keyed by ACCOUNT LOCATOR: inside a share, CURRENT_ACCOUNT() returns the consumer's locator
CREATE TABLE IF NOT EXISTS SHARING_DEMO_DB.SHARED.CONSUMER_ENTITLEMENTS (CONSUMER_LOCATOR STRING, CONSUMER_LABEL STRING, REGION STRING);
INSERT OVERWRITE INTO SHARING_DEMO_DB.SHARED.CONSUMER_ENTITLEMENTS
SELECT '{{CONSUMER_LOCATOR}}', '{{CONSUMER_ACCOUNT}}', 'APAC'                                              -- consumer: APAC only
UNION ALL
SELECT CURRENT_ACCOUNT(), CURRENT_ORGANIZATION_NAME() || '.' || CURRENT_ACCOUNT_NAME(), r.REGION   -- provider: everything
FROM (SELECT COLUMN1 AS REGION FROM VALUES ('APAC'), ('EMEA'), ('AMER')) r;
SELECT * FROM SHARING_DEMO_DB.SHARED.CONSUMER_ENTITLEMENTS;

In [ ]:
%%sql -r views
-- Idempotent: OR REPLACE + COPY GRANTS keeps existing privileges (including share grants) on re-run

-- Column level: no PII
CREATE OR REPLACE SECURE VIEW SHARING_DEMO_DB.SHARED.V_CUSTOMERS_NO_PII COPY GRANTS AS
SELECT CUSTOMER_ID, CUSTOMER_NAME, COUNTRY, SEGMENT
FROM SHARING_DEMO_DB.SALES.CUSTOMERS;

-- Static row level: APAC only
CREATE OR REPLACE SECURE VIEW SHARING_DEMO_DB.SHARED.V_ORDERS_APAC COPY GRANTS AS
SELECT ORDER_ID, CUSTOMER_ID, REGION, ORDER_DATE, AMOUNT_USD
FROM SHARING_DEMO_DB.SALES.ORDERS
WHERE REGION = 'APAC';

-- Per-consumer row level: rows depend on which account is querying
CREATE OR REPLACE SECURE VIEW SHARING_DEMO_DB.SHARED.V_ORDERS_ENTITLED COPY GRANTS AS
SELECT o.ORDER_ID, o.CUSTOMER_ID, o.REGION, o.ORDER_DATE, o.AMOUNT_USD
FROM SHARING_DEMO_DB.SALES.ORDERS o
WHERE EXISTS (
  SELECT 1 FROM SHARING_DEMO_DB.SHARED.CONSUMER_ENTITLEMENTS e
  WHERE e.REGION = o.REGION
    AND e.CONSUMER_LOCATOR = CURRENT_ACCOUNT()
);

## Step 3: Create the share and grant objects selectively
Grant order matters: **database, then schema, then objects**. Here is exactly what goes in and what stays out:

| Object | Shared? | Why |
|---|---|---|
| `HR` schema | No | Schema-level exclusion |
| `SALES.ORDERS` | Yes | Table-level inclusion |
| `SALES.CUSTOMERS` | No (only through the view) | Raw PII stays private |
| `SALES.INTERNAL_COSTS` | No | Table-level exclusion within a shared schema |
| `SHARED.V_CUSTOMERS_NO_PII` | Yes | Column level |
| `SHARED.V_ORDERS_APAC` | Yes | Row level |
| `SHARED.V_ORDERS_ENTITLED` | Yes | Per-consumer row level |
| `SHARED.CONSUMER_ENTITLEMENTS` | No | Kept internal |

In [ ]:
%%sql -r share_grants
-- Idempotent: IF NOT EXISTS keeps the share (and its consumers); GRANTs are no-ops if already granted
CREATE SHARE IF NOT EXISTS SHARING_DEMO_SHARE COMMENT = 'Granularity demo: table, column, row, per-consumer';

GRANT USAGE ON DATABASE SHARING_DEMO_DB TO SHARE SHARING_DEMO_SHARE;
GRANT USAGE ON SCHEMA SHARING_DEMO_DB.SALES  TO SHARE SHARING_DEMO_SHARE;
GRANT USAGE ON SCHEMA SHARING_DEMO_DB.SHARED TO SHARE SHARING_DEMO_SHARE;
-- NOTE: no grant on schema HR

-- Table level: one table out of three in SALES
GRANT SELECT ON TABLE SHARING_DEMO_DB.SALES.ORDERS TO SHARE SHARING_DEMO_SHARE;

-- Column / row / per-consumer level via secure views
GRANT SELECT ON VIEW SHARING_DEMO_DB.SHARED.V_CUSTOMERS_NO_PII TO SHARE SHARING_DEMO_SHARE;
GRANT SELECT ON VIEW SHARING_DEMO_DB.SHARED.V_ORDERS_APAC      TO SHARE SHARING_DEMO_SHARE;
GRANT SELECT ON VIEW SHARING_DEMO_DB.SHARED.V_ORDERS_ENTITLED  TO SHARE SHARING_DEMO_SHARE;

## Step 4: Add the consumer account
The provider and consumer must be in the same region to add the consumer directly.

> **Edition restriction.** If the provider account is **Business Critical** and the consumer is on a lower edition, Snowflake blocks the share by default with *"Sharing is not allowed from a provider account on BUSINESS_CRITICAL+ edition to target accounts on lower edition…"*. This guardrail stops regulated data (for example HIPAA or PCI) from reaching an account without the same controls.
>
> This demo uses dummy data, so the cell below overrides the check **for this share only** with `SHARE_RESTRICTIONS = FALSE`. That needs the `OVERRIDE SHARE RESTRICTIONS` privilege (ACCOUNTADMIN has it by default). Other shares on the account are unaffected. Don't do this for real regulated data unless the consumer has equivalent controls.

In [ ]:
-- Override the Business Critical -> lower-edition check FOR THIS SHARE ONLY (dummy, non-sensitive data).
-- Idempotent: re-adding an account that is already a consumer is a no-op.
ALTER SHARE SHARING_DEMO_SHARE ADD ACCOUNTS = {{CONSUMER_ACCOUNT}} SHARE_RESTRICTIONS = FALSE;

## Step 5: Verify the share contents
`SHOW GRANTS TO SHARE` lists **every** object the consumer can reach. `HR`, `CUSTOMERS`, `INTERNAL_COSTS` and `CONSUMER_ENTITLEMENTS` should not appear.

In [ ]:
%%sql -r share_contents
SHOW GRANTS TO SHARE SHARING_DEMO_SHARE;

In [ ]:
%%sql -r share_consumers
SHOW SHARES LIKE 'SHARING_DEMO_SHARE';

## Step 6: Preview what the consumer sees (provider-side simulation)
The session parameter `SIMULATED_DATA_SHARING_CONSUMER` makes secure views behave as if the named consumer account were running the query. It only affects secure views. The first query shows the provider's own unfiltered view for comparison.

In [ ]:
%%sql -r provider_view
ALTER SESSION UNSET SIMULATED_DATA_SHARING_CONSUMER;
SELECT REGION, COUNT(*) AS ORDERS, SUM(AMOUNT_USD) AS TOTAL_USD
FROM SHARING_DEMO_DB.SHARED.V_ORDERS_ENTITLED GROUP BY REGION ORDER BY REGION;

In [ ]:
-- Simulate the consumer by its account locator
ALTER SESSION SET SIMULATED_DATA_SHARING_CONSUMER = '{{CONSUMER_LOCATOR}}';
SELECT REGION, COUNT(*) AS ORDERS, SUM(AMOUNT_USD) AS TOTAL_USD
FROM SHARING_DEMO_DB.SHARED.V_ORDERS_ENTITLED GROUP BY REGION ORDER BY REGION;

In [ ]:
%%sql -r reset_sim
ALTER SESSION UNSET SIMULATED_DATA_SHARING_CONSUMER;

## Step 7: Run on the consumer side (in `<CONSUMER_ORG>.<CONSUMER_ACCOUNT>`)
Run this in the **consumer** account (ACCOUNTADMIN or a role with `IMPORT SHARE`). Replace `<PROVIDER_ORG>.<PROVIDER_ACCOUNT>` with the provider's identifier (the `org` and `account` values from the Step 0 context query).

```sql
-- 1. See the inbound share
SHOW SHARES LIKE 'SHARING_DEMO_SHARE';

-- 2. Mount it as a read-only database
CREATE DATABASE IF NOT EXISTS SHARING_DEMO_FROM_PROVIDER FROM SHARE <PROVIDER_ORG>.<PROVIDER_ACCOUNT>.SHARING_DEMO_SHARE;

-- 3. Schema level: HR is absent
SHOW SCHEMAS IN DATABASE SHARING_DEMO_FROM_PROVIDER;          -- SALES, SHARED (no HR)

-- 4. Table level: only ORDERS is in SALES
SHOW TABLES IN SCHEMA SHARING_DEMO_FROM_PROVIDER.SALES;       -- ORDERS only
SELECT * FROM SHARING_DEMO_FROM_PROVIDER.SALES.INTERNAL_COSTS; -- error: does not exist or not authorized

-- 5. Column level: no EMAIL / PHONE
SELECT * FROM SHARING_DEMO_FROM_PROVIDER.SHARED.V_CUSTOMERS_NO_PII;

-- 6. Static row level: APAC only
SELECT * FROM SHARING_DEMO_FROM_PROVIDER.SHARED.V_ORDERS_APAC;

-- 7. Per-consumer row level: this consumer sees only APAC
SELECT REGION, COUNT(*) FROM SHARING_DEMO_FROM_PROVIDER.SHARED.V_ORDERS_ENTITLED GROUP BY REGION;

-- 8. Secure view definitions are hidden
SELECT GET_DDL('VIEW','SHARING_DEMO_FROM_PROVIDER.SHARED.V_ORDERS_ENTITLED'); -- definition is not exposed
```

**Live-update test:** insert a new APAC order on the provider side, then re-run query 7 on the consumer side. It appears straight away, because no data was copied.

## Going further
- **Value-level masking:** attach a masking policy to a column (for example, partially mask email). The policy is evaluated for the consumer when the column is shared through a table or view.
- **Row access policies** can replace the entitlement-check view for policy-based per-consumer filtering.
- **Revoke anytime:** `REVOKE SELECT ON VIEW ... FROM SHARE ...` or `ALTER SHARE ... REMOVE ACCOUNTS = ...` takes effect immediately.

## Step 8: Clean up the lab
**Run this only when you have finished the demo.** It is destructive and removes everything this notebook created.

Order matters:
1. **Consumer side (in `<CONSUMER_ORG>.<CONSUMER_ACCOUNT>`):** drop the mounted database first:
   ```sql
   DROP DATABASE IF EXISTS SHARING_DEMO_FROM_PROVIDER;
   ```
2. **Provider side (cell below):** remove the consumer, drop the share, then drop the database. Dropping the share revokes the consumer's access immediately, even if step 1 was skipped.

Every statement uses `IF EXISTS`, so the cell is safe to re-run.

In [ ]:
-- DESTRUCTIVE: removes the share and all demo objects. Idempotent via IF EXISTS.
ALTER SESSION UNSET SIMULATED_DATA_SHARING_CONSUMER;
ALTER SHARE IF EXISTS SHARING_DEMO_SHARE REMOVE ACCOUNTS = {{CONSUMER_ACCOUNT}};
DROP SHARE IF EXISTS SHARING_DEMO_SHARE;
DROP DATABASE IF EXISTS SHARING_DEMO_DB;
SHOW SHARES LIKE 'SHARING_DEMO_SHARE';  -- expect no rows